In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"


In [2]:
import logging

loggers = [logging.getLogger(name) for name in logging.root.manager.loggerDict]
for logger in loggers:
    if "transformers" in logger.name.lower():
        logger.setLevel(logging.ERROR)

In [3]:
from models.data import ArabicSocialMediaDataModule
from farasa.ner import FarasaNamedEntityRecognizer
from farasa.pos import FarasaPOSTagger

In [4]:
farasa_pos_tagger = FarasaPOSTagger(interactive=True)
farasa_ner_recognizer = FarasaNamedEntityRecognizer(interactive=True)

[2025-09-08 12:33:29,175 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.
[2025-09-08 12:33:37,111 - farasapy_logger - WARNING]: Be careful with large lines as they may break on interactive mode. You may switch to Standalone mode for such cases.


In [5]:
# Initialize the data module
# data_module = HC3TextDataModule()
data_module = ArabicSocialMediaDataModule(
    text_transforms={
        "pos": farasa_pos_tagger.tag,
        "ner": farasa_ner_recognizer.recognize,
    },
)

data_module.setup()

In [6]:
# Define the model (you can switch between different models)

from models.models import LitXLMRobertaModelWithTextTransforms
import pytorch_lightning as pl
from pytorch_lightning.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
)


In [ ]:
class CrossModelExperiment:
    def __init__(self, max_epochs=100, fit_model=True):
        self.max_epochs = max_epochs
        self.fit_model = fit_model
        self.results = {}
        self.chcekpoints_path = "trained_detectors/Arabic/ArabicSocialMediaDataset/{train_model}AIDetectorWithPOSAndNER/checkpoints"

    def _get_callbacks(self, train_model):
        early_stopping = EarlyStopping(
            monitor="val_loss",
            min_delta=0.0,
            patience=3,
            verbose=True,
            mode="min",
        )

        checkpoint = ModelCheckpoint(
            monitor="val_loss",
            dirpath=self.chcekpoints_path.format(train_model=train_model.title()),
            filename="best-checkpoint",
            save_top_k=1,
            mode="min",
        )

        return [early_stopping, checkpoint]

    def _test_on_model(self, trainer, model, test_model, train_model):
        # Load the best checkpoint before testing
        checkpoint_path = self.chcekpoints_path.format(train_model=train_model.title())
        checkpoint_path += "/best-checkpoint.ckpt"
        model = LitXLMRobertaModelWithTextTransforms.load_from_checkpoint(
            checkpoint_path
        )

        test_datamodule = ArabicSocialMediaDataModule(
            models=[test_model],
            text_transforms={
                "pos": farasa_pos_tagger.tag,
                "ner": farasa_ner_recognizer.recognize,
            },
        )
        test_datamodule.setup()

        results = trainer.test(model, test_datamodule.test_dataloader())[0]
        return {
            "accuracy": results["test_acc"],
            "precision": results["test_precision"],
            "recall": results["test_recall"],
            "f1": results["test_f1"],
            "loss": results["test_loss"],
        }

    def run_experiment(self, train_model, test_models):
        # Initialize components
        model = LitXLMRobertaModelWithTextTransforms()
        train_datamodule = ArabicSocialMediaDataModule(
            models=[train_model],
            text_transforms={
                "pos": farasa_pos_tagger.tag,
                "ner": farasa_ner_recognizer.recognize,
            },
        )
        trainer = pl.Trainer(
            devices=1,
            max_epochs=self.max_epochs,
            accelerator="auto",
            val_check_interval=0.25,
            check_val_every_n_epoch=1,
            callbacks=self._get_callbacks(train_model),
        )

        # Train the model
        if self.fit_model:
            print(f"\nTraining on {train_model} data...")
            trainer.fit(model, train_datamodule)

        # Test on all specified models
        results = {}
        for test_model in test_models:
            print(f"\nTesting on {test_model} data...")
            results[test_model] = self._test_on_model(
                trainer,
                model,
                test_model,
                train_model,
            )

        # Store results
        self.results[train_model] = results

        # Display results
        self._display_results(train_model, results)

        # Print checkpoint location
        checkpoint_dir = self.chcekpoints_path.format(train_model=train_model.title())
        print(
            f"\nBest model checkpoint saved at: {checkpoint_dir}/best-checkpoint.ckpt"
        )

        return results

    def _display_results(self, train_model, results):
        print(f"\nResults for model trained on {train_model}:")
        print("-" * 80)
        print(
            f"{'Test Model':<15} {'Accuracy':<10} {'Precision':<10} {'Recall':<10} {'F1':<10} {'Loss':<10}"
        )

        print("-" * 80)
        for test_model, metrics in results.items():
            print(
                f"{test_model:<15}"
                f"{metrics['accuracy']:<10.4f}"
                f"{metrics['precision']:<10.4f}"
                f"{metrics['recall']:<10.4f}"
                f"{metrics['f1']:<10.4f}"
                f"{metrics['loss']:<10.4f}"
            )
        print("-" * 80)

In [ ]:
import shutil
import json

available_models = ["allam", "jais-batched", "llama-batched", "openai"]
# available_models = ["allam"]
experiment = CrossModelExperiment(fit_model=True)
number_of_runs = 10

all_results = {}
for train_model in available_models:
    for i in range(number_of_runs):
        print(f"\n{'=' * 50}")
        print(f"\nRun {i + 1}/{number_of_runs} for training model: {train_model}")
        print(f"{'=' * 50}")
        results = experiment.run_experiment(
            train_model=train_model, test_models=available_models
        )
        if train_model not in all_results:
            all_results[train_model] = []
        all_results[train_model].append(results)
        # delete checkpoint directory to save space
        print("Deleting checkpoint directory to save space...")
        shutil.rmtree(
            experiment.chcekpoints_path.format(train_model=train_model.title()).removesuffix(
                "/best-checkpoint.ckpt",
            ),
            ignore_errors=True,
        )

In [ ]:
with open(
    "notebooks/Arabic_experiments/ArabicSocialMediaDataset/cross_model_detection_with_pos_ner_multiple_runs_results.json",
    "w",
) as f:
    json.dump(
        all_results,
        f,
        indent=4,
        ensure_ascii=False,
        default=str,
    )